# EXP-041 | CAT(ITE 신규) + ITE 확장(6→10쌍) × 5 seeds

지금까지 **CAT는 한 번도 ITE 피처를 사용하지 않았음.** XGB+ITE가 성공한 이유를 CAT에 적용.
동시에 ITE 쌍을 6개 → 10개로 확장해 새로운 범주형 상호작용 포착.

| 모델 | 피처셋 | 상태 |
|---|---|---|
| LGB | FE-v1 (85개) | 기존 유지 |
| CAT | FE-v2+TE (98개) | 기존 유지 |
| **CAT** | **FE-v2+TE+ITE_10 (108개)** | **신규 — CAT ITE 첫 시도** |
| XGB | FE-v2+TE+ITE_10 (108개) | 기존 6쌍 → **10쌍으로 확장** |

**신규 ITE 4쌍:**
- (나이, 총임신횟수): 나이 × 과거 임신 이력
- (나이, 총시술횟수): 나이 × 시술 경험 누적
- (배란유도유형, 난자출처): 자극 방식 × 난자 출처
- (특정시술유형, 난자출처): 세부 시술 × 난자 출처

| 기준선 | EXP-038 OOF 0.74090 (5seeds×3models) |

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from datetime import date
from scipy.stats import rankdata

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, recall_score, precision_score, accuracy_score
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
import xgboost as xgb

from src.preprocessing import preprocess

warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/raw')
OUT_DIR  = Path('../data/submissions')
DOCS_DIR = Path('../docs')
TARGET   = '임신 성공 여부'
N_FOLDS  = 5
SEEDS    = [42, 0, 123, 2024, 777]
EXP_NO   = 41
AUTHOR   = '조여진'

train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
sub   = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print(f'train: {train.shape}  /  test: {test.shape}')
print(f'Seeds: {SEEDS}  →  총 {len(SEEDS)*4*N_FOLDS}회 학습  (4모델×{len(SEEDS)}시드×{N_FOLDS}폴드)')

train: (256351, 69)  /  test: (90067, 68)
Seeds: [42, 0, 123, 2024, 777]  →  총 100회 학습  (4모델×5시드×5폴드)


## 1. 피처 준비

In [2]:
def add_pre_encode_features(df):
    df = df.copy()
    df['기증_난자_여부'] = (df['난자 출처'] == '기증 제공').astype(int)
    df['기증_정자_여부'] = df['정자 출처'].isin(['기증 제공', '배우자 및 기증 제공']).astype(int)
    return df

def add_derived_features_v1(df):
    df = df.copy()
    eps = 1e-6
    df['수정률']    = df['총 생성 배아 수']   / (df['혼합된 난자 수'] + eps)
    df['이식률']    = df['이식된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['저장률']    = df['저장된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['ICSI_비율'] = df['미세주입된 난자 수'] / (df['혼합된 난자 수'] + eps)
    df['배아_발달일']    = df['배아 이식 경과일'] - df['난자 혼합 경과일']
    df['신선_시술_여부']  = df['수집된 신선 난자 수'].notna().astype(int)
    male_cols   = ['남성 주 불임 원인','남성 부 불임 원인','불임 원인 - 남성 요인']
    female_cols = ['여성 주 불임 원인','여성 부 불임 원인','불임 원인 - 난관 질환',
                   '불임 원인 - 배란 장애','불임 원인 - 자궁내막증','불임 원인 - 자궁경부 문제']
    couple_cols = ['부부 주 불임 원인','부부 부 불임 원인']
    sperm_cols  = ['불임 원인 - 정자 농도','불임 원인 - 정자 운동성',
                   '불임 원인 - 정자 형태','불임 원인 - 정자 면역학적 요인']
    all_cause   = male_cols + female_cols + couple_cols + sperm_cols + ['불명확 불임 원인']
    df['남성_불임_합계']      = df[male_cols].sum(axis=1)
    df['여성_불임_합계']      = df[female_cols].sum(axis=1)
    df['부부_불임_합계']      = df[couple_cols].sum(axis=1)
    df['정자_문제_합계']      = df[sperm_cols].sum(axis=1)
    df['총_불임원인_수']      = df[all_cause].sum(axis=1)
    df['임신시도기록_있음']    = df['임신 시도 또는 마지막 임신 경과 연수'].notna().astype(int)
    df['신선_난자_저장_있음']  = (df['저장된 신선 난자 수'] > 0).astype(int)
    df['나이_시술횟수_상호작용'] = df['시술 당시 나이'] * df['총 시술 횟수']
    return df

def add_derived_features_v2(df):
    df = add_derived_features_v1(df)
    eps = 1e-6
    df['임신_성공률']   = df['총 임신 횟수'] / (df['총 시술 횟수'] + eps)
    df['시술_실패_횟수'] = (df['총 시술 횟수'] - df['총 임신 횟수']).clip(lower=0)
    egg_count = df['수집된 신선 난자 수'].fillna(-1)
    df['최적_난자수']    = ((egg_count >= 5) & (egg_count <= 15)).astype(int)
    df['나이_이식배아수'] = df['시술 당시 나이'] * df['이식된 배아 수'].fillna(0)
    return df

TE_COLS = ['시술 시기 코드','시술 유형','특정 시술 유형','배란 유도 유형',
           '난자 출처','정자 출처','시술 당시 나이','총 시술 횟수','총 임신 횟수']

# 기존 6쌍 + 신규 4쌍 = 10쌍
ITE_PAIRS_ORIG = [
    ('시술 당시 나이', '시술 유형'),
    ('시술 당시 나이', '특정 시술 유형'),
    ('시술 당시 나이', '난자 출처'),
    ('시술 당시 나이', '배란 유도 유형'),
    ('시술 유형',      '총 시술 횟수'),
    ('난자 출처',      '시술 유형'),
]
ITE_PAIRS_NEW = [
    ('시술 당시 나이', '총 임신 횟수'),   # 나이 × 과거 임신 이력
    ('시술 당시 나이', '총 시술 횟수'),   # 나이 × 시술 경험 누적
    ('배란 유도 유형', '난자 출처'),       # 자극 방식 × 난자 출처
    ('특정 시술 유형', '난자 출처'),       # 세부 시술 × 난자 출처
]
ITE_PAIRS_ALL = ITE_PAIRS_ORIG + ITE_PAIRS_NEW  # 10쌍

TE_ALPHA, ITE_ALPHA = 10, 20

train_fe = add_pre_encode_features(train)
test_fe  = add_pre_encode_features(test)
X_base_tr, X_base_te = preprocess(train_fe, test_fe)

X_v1_train = add_derived_features_v1(X_base_tr)
X_v1_test  = add_derived_features_v1(X_base_te)
X_v2_train = add_derived_features_v2(X_base_tr)
X_v2_test  = add_derived_features_v2(X_base_te)

# 컬럼 존재 여부 필터
TE_COLS        = [c for c in TE_COLS if c in X_v2_train.columns]
ITE_PAIRS_ORIG = [(c1,c2) for c1,c2 in ITE_PAIRS_ORIG if c1 in X_v2_train.columns and c2 in X_v2_train.columns]
ITE_PAIRS_NEW  = [(c1,c2) for c1,c2 in ITE_PAIRS_NEW  if c1 in X_v2_train.columns and c2 in X_v2_train.columns]
ITE_PAIRS_ALL  = ITE_PAIRS_ORIG + ITE_PAIRS_NEW

y_train = train[TARGET]
neg_pos_ratio = (y_train == 0).sum() / (y_train == 1).sum()

n_te_features  = len(TE_COLS)
n_ite_orig = len(ITE_PAIRS_ORIG)
n_ite_all  = len(ITE_PAIRS_ALL)
n_feat_cat = X_v2_train.shape[1] + n_te_features
n_feat_xgb = n_feat_cat + n_ite_all

print(f'FE-v1: {X_v1_train.shape[1]}개')
print(f'CAT(TE): {n_feat_cat}개  /  CAT(TE+ITE_10)=XGB: {n_feat_xgb}개')
print(f'TE: {n_te_features}개  /  ITE 기존: {n_ite_orig}쌍  /  ITE 신규: {len(ITE_PAIRS_NEW)}쌍  →  합계: {n_ite_all}쌍')
print(f'신규 ITE 쌍: {ITE_PAIRS_NEW}')

FE-v1: 85개
CAT(TE): 98개  /  CAT(TE+ITE_10)=XGB: 108개
TE: 9개  /  ITE 기존: 6쌍  /  ITE 신규: 4쌍  →  합계: 10쌍
신규 ITE 쌍: [('시술 당시 나이', '총 임신 횟수'), ('시술 당시 나이', '총 시술 횟수'), ('배란 유도 유형', '난자 출처'), ('특정 시술 유형', '난자 출처')]


## 2. 모델 파라미터

In [3]:
LGB_PARAMS_BASE = dict(
    objective='binary', metric='auc', verbosity=-1,
    num_threads=-1, is_unbalance=True, bagging_freq=1,
    learning_rate=0.035425966303284824,
    num_leaves=266, max_depth=5, min_child_samples=166,
    feature_fraction=0.5346439126449233,
    bagging_fraction=0.7122309235479091,
    lambda_l1=9.901034988600228,
    lambda_l2=2.213951873239442,
    min_gain_to_split=0.11418176854933762,
)
CAT_PARAMS_BASE = dict(
    iterations=2000, loss_function='Logloss', eval_metric='AUC',
    auto_class_weights='Balanced', thread_count=-1, verbose=False,
    early_stopping_rounds=100,
    learning_rate=0.018758723768855998, depth=6,
    l2_leaf_reg=9.189608434163782, min_data_in_leaf=19,
    subsample=0.8170921295501483, colsample_bylevel=0.6936810336930781,
)
XGB_PARAMS_BASE = dict(
    n_estimators=2000, scale_pos_weight=neg_pos_ratio,
    eval_metric='auc', tree_method='hist',
    n_jobs=-1, verbosity=0, early_stopping_rounds=100,
    learning_rate=0.05520069867907647, max_depth=4, min_child_weight=59,
    subsample=0.7663066457187595, colsample_bytree=0.6581836436885355,
    reg_alpha=8.692038126211928, reg_lambda=0.23932562420374562,
)

def rank_avg_normalize(arrays):
    ranks = np.stack([rankdata(a) for a in arrays], axis=1)
    avg = ranks.mean(axis=1)
    return (avg - avg.min()) / (avg.max() - avg.min())

print('파라미터 설정 완료')
print('모델: LGB(FE-v1) + CAT(FE-v2+TE) + CAT(FE-v2+TE+ITE_10) + XGB(FE-v2+TE+ITE_10)')

파라미터 설정 완료
모델: LGB(FE-v1) + CAT(FE-v2+TE) + CAT(FE-v2+TE+ITE_10) + XGB(FE-v2+TE+ITE_10)


## 3. Multi-seed 학습

4개 모델 × 5 seeds = 20모델 Rank Average.
TE/ITE는 fold 내부 계산 (data leakage 방지).

In [4]:
n_train = len(X_v1_train)
n_test  = len(X_v1_test)

all_oof_lgb   = []   # LGB  FE-v1
all_oof_cat1  = []   # CAT  FE-v2+TE
all_oof_cat2  = []   # CAT  FE-v2+TE+ITE_10  ← 신규
all_oof_xgb   = []   # XGB  FE-v2+TE+ITE_10
all_test_lgb  = []
all_test_cat1 = []
all_test_cat2 = []
all_test_xgb  = []

for s_idx, seed in enumerate(SEEDS, 1):
    print(f'\n{"="*60}')
    print(f'Seed {seed}  ({s_idx}/{len(SEEDS)})')
    print(f'{"="*60}')

    lgb_p = {**LGB_PARAMS_BASE, 'seed': seed}
    cat_p = {**CAT_PARAMS_BASE, 'random_seed': seed}
    xgb_p = {**XGB_PARAMS_BASE, 'random_state': seed}
    skf   = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

    oof_lgb  = np.zeros(n_train)
    oof_cat1 = np.zeros(n_train)
    oof_cat2 = np.zeros(n_train)
    oof_xgb  = np.zeros(n_train)
    test_lgb  = np.zeros(n_test)
    test_cat1 = np.zeros(n_test)
    test_cat2 = np.zeros(n_test)
    test_xgb  = np.zeros(n_test)

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_v1_train, y_train), 1):
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        global_mean = y_tr.mean()

        # ── [1] LGB: FE-v1 ──────────────────────────────────
        X_tr_lgb  = X_v1_train.iloc[tr_idx]
        X_val_lgb = X_v1_train.iloc[val_idx]
        ds_tr  = lgb.Dataset(X_tr_lgb, label=y_tr)
        ds_val = lgb.Dataset(X_val_lgb, label=y_val, reference=ds_tr)
        m = lgb.train(lgb_p, ds_tr, num_boost_round=2000, valid_sets=[ds_val],
                      callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)])
        oof_lgb[val_idx] = m.predict(X_val_lgb)
        test_lgb        += m.predict(X_v1_test) / N_FOLDS

        # ── TE 계산 (fold 내부) ─────────────────────────────
        X_tr_te  = X_v2_train.iloc[tr_idx].copy()
        X_val_te = X_v2_train.iloc[val_idx].copy()
        X_te_te  = X_v2_test.copy()
        for col in TE_COLS:
            grp = y_tr.groupby(X_tr_te[col])
            sm  = (grp.count() * grp.mean() + TE_ALPHA * global_mean) / (grp.count() + TE_ALPHA)
            tc  = f'{col}_te'
            X_tr_te[tc]  = X_tr_te[col].map(sm).fillna(global_mean)
            X_val_te[tc] = X_val_te[col].map(sm).fillna(global_mean)
            X_te_te[tc]  = X_te_te[col].map(sm).fillna(global_mean)

        # ── [2] CAT: FE-v2+TE (기존) ───────────────────────
        m = CatBoostClassifier(**cat_p)
        m.fit(X_tr_te, y_tr, eval_set=Pool(X_val_te, y_val), use_best_model=True)
        oof_cat1[val_idx] = m.predict_proba(X_val_te)[:, 1]
        test_cat1        += m.predict_proba(X_te_te)[:, 1] / N_FOLDS

        # ── ITE_10 계산 (fold 내부) ─────────────────────────
        X_tr_ite  = X_tr_te.copy()
        X_val_ite = X_val_te.copy()
        X_te_ite  = X_te_te.copy()
        for col1, col2 in ITE_PAIRS_ALL:
            key_tr  = X_tr_ite[col1].astype(str)  + '_' + X_tr_ite[col2].astype(str)
            key_val = X_val_ite[col1].astype(str) + '_' + X_val_ite[col2].astype(str)
            key_te  = X_te_ite[col1].astype(str)  + '_' + X_te_ite[col2].astype(str)
            grp = y_tr.groupby(key_tr)
            sm  = (grp.count() * grp.mean() + ITE_ALPHA * global_mean) / (grp.count() + ITE_ALPHA)
            ic  = f'{col1}_{col2}_ite'
            X_tr_ite[ic]  = key_tr.map(sm).fillna(global_mean)
            X_val_ite[ic] = key_val.map(sm).fillna(global_mean)
            X_te_ite[ic]  = key_te.map(sm).fillna(global_mean)

        # ── [3] CAT: FE-v2+TE+ITE_10 (신규) ────────────────
        m = CatBoostClassifier(**cat_p)
        m.fit(X_tr_ite, y_tr, eval_set=Pool(X_val_ite, y_val), use_best_model=True)
        oof_cat2[val_idx] = m.predict_proba(X_val_ite)[:, 1]
        test_cat2        += m.predict_proba(X_te_ite)[:, 1] / N_FOLDS

        # ── [4] XGB: FE-v2+TE+ITE_10 ───────────────────────
        m = xgb.XGBClassifier(**xgb_p)
        m.fit(X_tr_ite, y_tr, eval_set=[(X_val_ite, y_val)], verbose=False)
        oof_xgb[val_idx] = m.predict_proba(X_val_ite)[:, 1]
        test_xgb        += m.predict_proba(X_te_ite)[:, 1] / N_FOLDS

        f1 = roc_auc_score(y_val, oof_lgb[val_idx])
        f2 = roc_auc_score(y_val, oof_cat1[val_idx])
        f3 = roc_auc_score(y_val, oof_cat2[val_idx])
        f4 = roc_auc_score(y_val, oof_xgb[val_idx])
        print(f'  Fold {fold}  LGB={f1:.5f}  CAT1(TE)={f2:.5f}  CAT2(ITE)={f3:.5f}  XGB={f4:.5f}')

    seed_oof = rank_avg_normalize([oof_lgb, oof_cat1, oof_cat2, oof_xgb])
    seed_auc = roc_auc_score(y_train, seed_oof)
    cat2_auc = roc_auc_score(y_train, oof_cat2)
    print(f'  → Seed {seed} OOF (4모델 Rank Avg): {seed_auc:.5f}  |  CAT2 단독: {cat2_auc:.5f}')

    all_oof_lgb.append(oof_lgb)
    all_oof_cat1.append(oof_cat1)
    all_oof_cat2.append(oof_cat2)
    all_oof_xgb.append(oof_xgb)
    all_test_lgb.append(test_lgb)
    all_test_cat1.append(test_cat1)
    all_test_cat2.append(test_cat2)
    all_test_xgb.append(test_xgb)

print('\n학습 완료')


Seed 42  (1/5)
  Fold 1  LGB=0.73812  CAT1(TE)=0.73827  CAT2(ITE)=0.73820  XGB=0.73830
  Fold 2  LGB=0.74318  CAT1(TE)=0.74338  CAT2(ITE)=0.74314  XGB=0.74331
  Fold 3  LGB=0.74072  CAT1(TE)=0.74020  CAT2(ITE)=0.74031  XGB=0.74018
  Fold 4  LGB=0.73880  CAT1(TE)=0.73864  CAT2(ITE)=0.73879  XGB=0.73858
  Fold 5  LGB=0.74160  CAT1(TE)=0.74062  CAT2(ITE)=0.74087  XGB=0.74149
  → Seed 42 OOF (4모델 Rank Avg): 0.74067  |  CAT2 단독: 0.74026

Seed 0  (2/5)
  Fold 1  LGB=0.73952  CAT1(TE)=0.73945  CAT2(ITE)=0.73930  XGB=0.73948
  Fold 2  LGB=0.74057  CAT1(TE)=0.74013  CAT2(ITE)=0.74033  XGB=0.74060
  Fold 3  LGB=0.74097  CAT1(TE)=0.74146  CAT2(ITE)=0.74128  XGB=0.74123
  Fold 4  LGB=0.73973  CAT1(TE)=0.73962  CAT2(ITE)=0.73930  XGB=0.74013
  Fold 5  LGB=0.74002  CAT1(TE)=0.74054  CAT2(ITE)=0.74088  XGB=0.74069
  → Seed 0 OOF (4모델 Rank Avg): 0.74058  |  CAT2 단독: 0.74020

Seed 123  (3/5)
  Fold 1  LGB=0.73881  CAT1(TE)=0.73863  CAT2(ITE)=0.73877  XGB=0.73890
  Fold 2  LGB=0.74084  CAT1(TE)=0.74039

## 4. Rank Average — OOF & Test

In [5]:
# 20모델 rank avg (5seeds × 4models)
all_oofs  = all_oof_lgb  + all_oof_cat1  + all_oof_cat2  + all_oof_xgb
all_tests = all_test_lgb + all_test_cat1 + all_test_cat2 + all_test_xgb

oof_final  = rank_avg_normalize(all_oofs)
test_final = rank_avg_normalize(all_tests)

auc_final = roc_auc_score(y_train, oof_final)

BASELINE_032 = 0.74071
BASELINE_038 = 0.74090
GAP          = 0.00148

print(f'[EXP-041] OOF AUC (20모델 Rank Avg): {auc_final:.5f}')
print(f'  vs EXP-032: {auc_final - BASELINE_032:+.5f}')
print(f'  vs EXP-038: {auc_final - BASELINE_038:+.5f}')
print(f'  예상 제출 점수: {auc_final + GAP:.5f}')

# CAT2(ITE) 기여 분석
oof_3m = rank_avg_normalize(all_oof_lgb + all_oof_cat1 + all_oof_xgb)
auc_3m = roc_auc_score(y_train, oof_3m)
print(f'\n[비교] 3모델(CAT2 제외): {auc_3m:.5f}  →  4모델: {auc_final:.5f}  (CAT2 기여: {auc_final-auc_3m:+.5f})')

# XGB ITE 확장 효과 (CAT2 없이 XGB_10 vs XGB_6 비교)
# 각 시드별 CAT2 단독 OOF AUC
print('\n시드별 OOF AUC:')
for i, seed in enumerate(SEEDS):
    sa4 = roc_auc_score(y_train, rank_avg_normalize(
        [all_oof_lgb[i], all_oof_cat1[i], all_oof_cat2[i], all_oof_xgb[i]]))
    cat2_indiv = roc_auc_score(y_train, all_oof_cat2[i])
    print(f'  Seed {seed:>4d}:  4모델={sa4:.5f}  CAT2(ITE)단독={cat2_indiv:.5f}')

[EXP-041] OOF AUC (20모델 Rank Avg): 0.74086
  vs EXP-032: +0.00015
  vs EXP-038: -0.00004
  예상 제출 점수: 0.74234

[비교] 3모델(CAT2 제외): 0.74089  →  4모델: 0.74086  (CAT2 기여: -0.00003)

시드별 OOF AUC:
  Seed   42:  4모델=0.74067  CAT2(ITE)단독=0.74026
  Seed    0:  4모델=0.74058  CAT2(ITE)단독=0.74020
  Seed  123:  4모델=0.74053  CAT2(ITE)단독=0.74022
  Seed 2024:  4모델=0.74076  CAT2(ITE)단독=0.74040
  Seed  777:  4모델=0.74069  CAT2(ITE)단독=0.74030


## 5. Submission 저장 & 실험 기록

In [6]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

auc_str   = f'{auc_final:.4f}'.replace('.', '')
out_fname = f'submission_exp{EXP_NO:03d}_{AUTHOR}_{auc_str}.csv'
sub_out   = sub.copy()
sub_out['probability'] = test_final
sub_out.to_csv(OUT_DIR / out_fname, index=False)
print(f'저장: {out_fname}')

저장: submission_exp041_조여진_07409.csv


In [7]:
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill

def log_to_leaderboard(exp_no, author, model_name, params_str,
                        f1, recall, precision, accuracy, oof_auc,
                        cv_strategy, preprocessing_ver, n_features,
                        imbalance_method, submitted, hackathon_score,
                        file_name, notes='', insights=''):
    lb_path = DOCS_DIR / 'leaderboard.xlsx'
    wb = load_workbook(lb_path)
    ws = wb['리더보드']
    exp_label = f'EXP-{exp_no:03d}'
    next_row = ws.max_row + 1
    for r in range(2, ws.max_row + 1):
        val = ws.cell(row=r, column=2).value
        if val == exp_label:
            next_row = r; break
        if ws.cell(row=r, column=1).value is None or str(ws.cell(row=r, column=1).value).strip() == '':
            next_row = r; break
    values = [str(date.today()), exp_label, author, model_name, params_str,
              round(f1,5), round(recall,5), round(precision,5), round(accuracy,5), round(oof_auc,5),
              cv_strategy, preprocessing_ver, n_features, imbalance_method,
              submitted, hackathon_score, file_name, notes, insights]
    thin   = Side(style='thin', color='B0B8D0')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    fill   = PatternFill('solid', fgColor='EEF2FA') if next_row % 2 == 0 else None
    font   = Font(name='맑은 고딕', size=10)
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    left   = Alignment(horizontal='left',   vertical='center', wrap_text=True)
    left_cols = {4, 5, 12, 14, 17, 18, 19}
    for c_idx, val in enumerate(values, start=1):
        cell = ws.cell(row=next_row, column=c_idx, value=val)
        cell.font = font; cell.border = border
        cell.alignment = left if c_idx in left_cols else center
        if fill: cell.fill = fill
        if c_idx in range(6, 11) or c_idx == 16: cell.number_format = '0.00000'
    wb.save(lb_path)
    print(f'[leaderboard.xlsx] EXP-{exp_no:03d} 기록 완료 (row {next_row})')

oof_binary = (oof_final >= 0.5).astype(int)
params_str = (f'MultiSeed(5) × 4모델 | ITE 6→10쌍 확장 | seeds={SEEDS} | '
              f'LGB(FE-v1)+CAT(TE)+CAT(TE+ITE_10)+XGB(TE+ITE_10) | 20모델 RankAvg')
NOTES    = ('CAT에 ITE 첫 적용. ITE 6쌍→10쌍 확장(신규: 나이×총임신횟수, 나이×총시술횟수, '
            '배란유도유형×난자출처, 특정시술유형×난자출처). '
            '4모델(LGB-v1+CAT-TE+CAT-ITE10+XGB-ITE10)×5seeds=20모델 RankAvg.')
INSIGHTS = (f'EXP-038(5seeds 3models) OOF={BASELINE_038:.5f} 대비 {auc_final-BASELINE_038:+.5f} | '
            f'CAT2(ITE) 기여: {auc_final-auc_3m:+.5f} | '
            f'예상 제출: {auc_final+GAP:.5f}')

log_to_leaderboard(
    EXP_NO, AUTHOR, 'CAT-ITE+ITE10+MultiSeed5(LGB+CAT×2+XGB)', params_str,
    f1_score(y_train, oof_binary), recall_score(y_train, oof_binary),
    precision_score(y_train, oof_binary), accuracy_score(y_train, oof_binary),
    auc_final, f'Stratified {N_FOLDS}-Fold × {len(SEEDS)} seeds',
    'mixed(v1/v2+TE+ITE10)', X_v1_train.shape[1],
    'is_unbalance / Balanced / scale_pos_weight',
    'N', None, f'notebooks/37_cat_ite_expanded_yjcho.ipynb', NOTES, INSIGHTS
)

[leaderboard.xlsx] EXP-041 기록 완료 (row 38)
